[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EddiazOS/LC_data_review/blob/main/05_Analisis_Reproducible/notebooks/03_nivel_3_analisis_multivariado.ipynb)

# Notebook 03: Nivel 3 — Análisis Multivariado y Fuentes de Emisión
### Subsección 3.3 del Manuscrito: Correlación de Spearman, PCA, Propiedades Fisicoquímicas
**Proyecto LC-GC / Indoor Dust — Universidad de Cartagena & Universidad de Amberes**


In [ ]:
# Celda 1: Configuración de Entorno (Google Colab / Local) y Pre-requisitos
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("🚀 Entorno Google Colab detectado.")
    if not os.path.exists('LC_data_review'):
        !git clone https://github.com/EddiazOS/LC_data_review.git
    if os.path.exists('LC_data_review/05_Analisis_Reproducible/datos_curados'):
        DATA_DIR = 'LC_data_review/05_Analisis_Reproducible/datos_curados'
        OUT_TABLES = 'LC_data_review/05_Analisis_Reproducible/outputs/tablas'
        OUT_FIGS = 'LC_data_review/05_Analisis_Reproducible/outputs/figuras'
    else:
        DATA_DIR = '../datos_curados'
        OUT_TABLES = '../outputs/tablas'
        OUT_FIGS = '../outputs/figuras'
else:
    print("💻 Entorno Local detectado.")
    DATA_DIR = '../datos_curados' if os.path.exists('../datos_curados') else '05_Analisis_Reproducible/datos_curados'
    OUT_TABLES = '../outputs/tablas' if os.path.exists('../outputs/tablas') else '05_Analisis_Reproducible/outputs/tablas'
    OUT_FIGS = '../outputs/figuras' if os.path.exists('../outputs/figuras') else '05_Analisis_Reproducible/outputs/figuras'

os.makedirs(OUT_TABLES, exist_ok=True)
os.makedirs(OUT_FIGS, exist_ok=True)
print(f"Directorio de datos activo: {DATA_DIR}")

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
print("Librerías cargadas correctamente.")


In [ ]:
# Celda 2: Auto-Carga de Datos Curados
df_muestras = pd.read_csv(os.path.join(DATA_DIR, 'muestras_metadata.csv'))
df_compuestos = pd.read_csv(os.path.join(DATA_DIR, 'compuestos_metadata.csv'))
df_conc = pd.read_csv(os.path.join(DATA_DIR, 'concentraciones_ng_g.csv'))
df_det = pd.read_csv(os.path.join(DATA_DIR, 'estado_deteccion.csv'))

acronyms = list(df_compuestos['acronym'])
print(f"Cargadas {len(df_muestras)} muestras y {len(acronyms)} compuestos.")


In [ ]:
# Celda 3: Matriz de Correlación de Spearman y Pares Significativos (Tabla 4)
valid_analytes = [c for c in acronyms if (df_det[c].sum() / 44.0) >= 0.20]
df_sub = df_conc[valid_analytes].copy()

corr_matrix, p_matrix = stats.spearmanr(df_sub)
df_corr = pd.DataFrame(corr_matrix, index=valid_analytes, columns=valid_analytes)
df_p = pd.DataFrame(p_matrix, index=valid_analytes, columns=valid_analytes)

sig_pairs = []
for i in range(len(valid_analytes)):
    for j in range(i + 1, len(valid_analytes)):
        c1 = valid_analytes[i]
        c2 = valid_analytes[j]
        r_val = df_corr.iloc[i, j]
        p_val = df_p.iloc[i, j]
        if r_val >= 0.50 and p_val < 0.05:
            sig_pairs.append({
                'Compound_1': c1,
                'Compound_2': c2,
                'Spearman_r': r_val,
                'p_value': p_val
            })

df_sig_pairs = pd.DataFrame(sig_pairs).sort_values(by='Spearman_r', ascending=False)
df_sig_pairs.to_csv(os.path.join(OUT_TABLES, 'tabla_4_correlaciones_spearman_significativas.csv'), index=False)
display(df_sig_pairs.head(15))


In [ ]:
# Celda 4: Generación de la Figura S1 (Heatmap de Spearman)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(df_corr, cmap='RdBu_r', center=0, vmin=-0.5, vmax=1.0, annot=False, square=True,
            cbar_kws={'label': "Spearman rank correlation coefficient (r)"}, ax=ax)
ax.set_title("Bivariate Spearman Rank Correlation Matrix (Indoor Dust)", fontweight='bold', pad=12)

s1_path = os.path.join(OUT_FIGS, 'figura_s1_spearman_heatmap.png')
plt.savefig(s1_path)
plt.show()
print(f"Figura S1 guardada en: {s1_path}")


In [ ]:
# Celda 5: Análisis de Componentes Principales (PCA)
X_raw = df_sub.values
X_log = np.log10(X_raw + 1.0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

pca = PCA(n_components=5)
scores = pca.fit_transform(X_scaled)

exp_var = pca.explained_variance_ratio_ * 100
print(f"Varianza explicada: PC1 = {exp_var[0]:.2f}%, PC2 = {exp_var[1]:.2f}%, PC3 = {exp_var[2]:.2f}%")
print(f"Varianza acumulada (PC1 + PC2): {exp_var[0] + exp_var[1]:.2f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = {'Car': '#E67E22', 'House': '#27AE60', 'Office': '#2980B9'}
for micro in ['Car', 'House', 'Office']:
    mask = (df_muestras['microenvironment'] == micro).values
    ax1.scatter(scores[mask, 0], scores[mask, 1], label=micro, color=colors[micro], s=60, alpha=0.85)

ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax1.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel(f"PC1 ({exp_var[0]:.1f}%)", fontweight='bold')
ax1.set_ylabel(f"PC2 ({exp_var[1]:.1f}%)", fontweight='bold')
ax1.set_title("(a) PCA Scores by Microenvironment", fontweight='bold')
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.5)

loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
for i, comp in enumerate(valid_analytes):
    ax2.arrow(0, 0, loadings[i, 0], loadings[i, 1], color='#2C3E50', alpha=0.6, head_width=0.03)
    ax2.text(loadings[i, 0]*1.1, loadings[i, 1]*1.1, comp, fontsize=7.5)

ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax2.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel(f"PC1 ({exp_var[0]:.1f}%)", fontweight='bold')
ax2.set_ylabel(f"PC2 ({exp_var[1]:.1f}%)", fontweight='bold')
ax2.set_title("(b) Variable Loadings Vectors", fontweight='bold')
ax2.grid(True, linestyle=':', alpha=0.5)

pca_path = os.path.join(OUT_FIGS, 'figura_4_pca_composite.png')
plt.savefig(pca_path)
plt.show()
print(f"Figura 4 guardada en: {pca_path}")
